# 🐍 Python Level 1 — Bonus Demo
## 🏦 Natixis Trading Floor — Rookie Trader, 12 Months

**A live showcase for the end of Class 3**

---

### 🎉 You've finished Level 1. Now look what you can already build.

Over three classes you picked up variables, loops, `if` / `else`, lists, tuples, sets,
dictionaries, and finally pandas. That is already enough to build something that *feels*
like a real application — not a toy exercise, a small trading game with a market, news
headlines, a trading book, and a pandas debrief at the end.

**Before the mini-project starts, run this notebook top to bottom and watch it work.**

### 🕹️ The rules of the game

- You start with **100,000 EUR** in cash and no positions.
- Every month, one **headline** moves one asset, and every asset also wobbles a little on
  its own.
- After seeing the prices, you type a command:
  - `buy TICKER QTY` — e.g. `buy BANK 10`
  - `sell TICKER QTY` — e.g. `sell GOLD 5`
  - `hold` — do nothing this month
- After 12 months, we compare what you ended up with against a **benchmark**: someone who
  bought a bit of everything on day one and never touched it again.

### ✅ No new syntax

Every mechanic below — the market, the trading loop, the debrief — is built only from what
you already know from Classes 1 to 3. Two small things are genuinely new (`import random`,
and one bonus chart at the very end); both are called out clearly, right where they happen.

---
## 1. ⚙️ Setting up

Every script starts with its **settings** — the knobs you can turn without touching the
logic below. Here that means: do we auto-play the demo or let a student type moves, how
much cash we start with, and how many months we simulate.

> 🆕 **New today: `import random`**
>
> `random` is a library, just like `pandas` — you bring it in with `import` and it comes
> built into Python, nothing to install. We only need three things from it:
> - `random.seed(42)` — locks the "randomness" so it produces the **same** sequence every
>   time you run it. Without this, the demo would give different numbers on every run.
> - `random.choice(a_list)` — picks one random item out of a list.
> - `random.uniform(low, high)` — picks a random decimal number between `low` and `high`.

Run the cell below once. `AUTO_PLAY = True` means the notebook plays the whole game itself
with a scripted set of moves — perfect for a live demo. Flip it to `False` and re-run if
you want a student to type the moves live with `input()`.

In [ ]:
# ── SETTINGS ────────────────────────────────────────────────────────────────
import random          # 🆕 new library this class - see the markdown above
import pandas as pd    # the pandas import from Class 3

AUTO_PLAY  = True       # True  = the notebook plays itself (great for a live demo)
                        # False = you get an input() prompt and play for real
SEED       = 42         # locks the "random" numbers so this demo is reproducible
START_CASH = 100000     # EUR, no positions to start
MONTHS     = 12         # one trading year

random.seed(SEED)       # from this point on, "random" choices follow a fixed sequence

---
## 2. 📈 The market

The market is a **dictionary of dictionaries** — nested exactly one level deep, exactly
what Class 2 covered. The outer key is the ticker (`"OATS"`, `"BANK"`...), and the value is
another dictionary holding that asset's name, current price, and how much it typically
swings in a month.

`headlines` is a **list of tuples**. Each tuple packs together a news sentence, which
ticker it hits, and by how much — later we unpack all three at once with
`news, hit_ticker, boost = random.choice(headlines)` (tuple unpacking, Class 2).

`script` is simply the list of moves `AUTO_PLAY` will "type" for you, one string per
month — plain strings, split apart later the same way a real command would be.

In [ ]:
# ── THE MARKET ──────────────────────────────────────────────────────────────
# A dictionary of dictionaries - nested one level deep (Class 2)
assets = {
    "OATS": {"name": "French 10Y government bonds", "price": 100.00, "swing": 0.02},
    "BANK": {"name": "European banks index",        "price":  48.50, "swing": 0.07},
    "GOLD": {"name": "Gold ETF",                    "price": 210.00, "swing": 0.04},
    "MOON": {"name": "Crypto (unregulated)",        "price":  15.00, "swing": 0.25},
}

# Remember day-one prices - the game cell below needs to restore them every time you
# re-run it, since the trading loop mutates the prices inside `assets` in place.
original_prices = {}
for ticker in assets:
    original_prices[ticker] = assets[ticker]["price"]

# A list of tuples: (headline text, ticker it hits, % move that month)
headlines = [
    ("The ECB leaves rates unchanged. Bond desks exhale.",          "OATS",  0.03),
    ("Inflation surprises on the upside. Bonds sell off.",          "OATS", -0.04),
    ("Stress-test results are better than feared.",                 "BANK",  0.09),
    ("A mid-size lender misses its provisions target.",             "BANK", -0.10),
    ("Geopolitical tension sends money into safe havens.",          "GOLD",  0.07),
    ("Risk appetite returns. Gold looks boring again.",             "GOLD", -0.05),
    ("An influencer calls it 'the future of money'.",               "MOON",  0.35),
    ("A regulator opens an investigation. Panic selling.",          "MOON", -0.30),
    ("Quiet month. Everyone is on holiday.",                        "OATS",  0.00),
]

# The scripted moves AUTO_PLAY "types" for you, month by month - plain strings for now,
# .split() turns each one into words a little further down.
script = [
    "buy OATS 200", "buy BANK 400", "buy GOLD 50",  "buy MOON 500",
    "hold",         "sell MOON 250","buy BANK 200", "hold",
    "sell GOLD 25", "buy MOON 300", "sell BANK 300","sell MOON 550",
]

---
## 3. 🎮 The game

Before you run it, here is what the loop below actually does, month by month — every
piece maps to something you already know:

1. **Remember last month's prices** — a `for` loop over `assets` (Class 2 dictionaries).
2. **Pick a headline** — `random.choice(headlines)`, then unpack it into three variables
   in one line (tuple unpacking, Class 2).
3. **Move every price** — `random.uniform(-swing, swing)` for the monthly wobble, plus
   the headline's `boost` if that ticker was hit. A price is never allowed to drop below
   0.50 (a plain `if`, Class 1).
4. **Print the price board** — an f-string with format specifiers (`{price:>9,.2f}`,
   Class 1), plus a 🟢▲ / 🔴▼ arrow decided with a plain `if` / `else`.
5. **Get a move** — either `input()` (Class 1) or the next line of `script` when
   `AUTO_PLAY` is on. `.split()` breaks `"buy BANK 10"` into `["buy", "bank", "10"]` (a
   string method), and a `while True` loop with `break` / `continue` (Class 1) keeps
   asking until the move makes sense: the right number of words, a real ticker
   (`in` / `not in`, Class 1), a quantity that `.isdigit()` (another string method) and
   casts cleanly to a whole number, and enough cash or enough units held.
6. **Apply the trade** — update `cash` and `portfolio` (a dictionary, Class 2), and
   `.append()` a record of it to `trade_log` (a list, Class 2) — this list is exactly
   what turns into a DataFrame in the debrief.
7. **Log the month** — cash, holdings, net worth and the passive benchmark all get
   `.append()`-ed to `history`, one dictionary per month.

The cell resets everything at the top (`cash`, `portfolio`, `trade_log`, `history`, the
random sequence, and the prices themselves), so you can run it again and again with
identical results — try running it twice in a row.

In [ ]:
# ── THE GAME ────────────────────────────────────────────────────────────────
# Re-runnable: everything the loop below changes gets reset right here, so running this
# cell twice in a row gives you the exact same 12 months both times.
random.seed(SEED)                                   # replay the same "random" sequence
for ticker in assets:
    assets[ticker]["price"] = original_prices[ticker]   # undo last run's price moves

cash      = START_CASH
portfolio = {"OATS": 0, "BANK": 0, "GOLD": 0, "MOON": 0}   # a dictionary, Class 2
trade_log = []                             # every executed trade, one dict per trade
history   = []                             # one snapshot per month

# The benchmark: split START_CASH equally across the 4 assets on day one, buy as many
# units as that allows, then never touch it again - "doing nothing" as a strategy.
start_prices    = {}
benchmark_units = {}
for ticker in assets:
    start_prices[ticker]    = assets[ticker]["price"]
    benchmark_units[ticker] = (START_CASH / len(assets)) / assets[ticker]["price"]

print("=" * 62)
print("        NATIXIS TRADING FLOOR - ROOKIE TRADER, 12 MONTHS")
print("=" * 62)
print(f"You start with {START_CASH:,.2f} EUR and no positions.")
print("Type:  buy TICKER QTY   |   sell TICKER QTY   |   hold")

month = 1
while month <= MONTHS:                     # one iteration per month (Class 1 while loop)

    last_prices = {}
    for ticker in assets:
        last_prices[ticker] = assets[ticker]["price"]

    # Tuple unpacking (Class 2): one random.choice() call, three variables at once
    news, hit_ticker, boost = random.choice(headlines)

    for ticker in assets:
        swing  = assets[ticker]["swing"]
        move   = random.uniform(-swing, swing)      # the everyday wobble
        if ticker == hit_ticker:
            move = move + boost                     # plus the headline, if it lands here
        new_price = assets[ticker]["price"] * (1 + move)
        if new_price < 0.5:                         # a price can never go to zero or below
            new_price = 0.5
        assets[ticker]["price"] = round(new_price, 2)

    holdings_value = 0
    for ticker in portfolio:
        holdings_value = holdings_value + portfolio[ticker] * assets[ticker]["price"]
    net_worth = cash + holdings_value

    print("")
    print("-" * 62)
    print(f"MONTH {month} of {MONTHS}   |   cash {cash:>12,.2f} EUR   |   net worth {net_worth:>12,.2f} EUR")
    print("-" * 62)
    print(f"NEWS: {news}")
    print("")
    for ticker in assets:
        name   = assets[ticker]["name"]
        price  = assets[ticker]["price"]
        before = last_prices[ticker]
        change = (price - before) / before * 100
        if change >= 0:
            arrow = "🟢▲"                             # up (or flat) this month
        else:
            arrow = "🔴▼"                             # down this month
        held = portfolio[ticker]
        print(f"  {ticker}  {name:<30} {price:>9,.2f}  {arrow} {change:>6.2f}%   held: {held}")
    print("")

    while True:                                     # keep asking until we get a valid move
        if AUTO_PLAY:
            command = script[month - 1]
            print(f"[auto-pilot] {command}")
        else:
            command = input("Your move > ")

        # .split() is a string method: it breaks the text on spaces into a list of words,
        # e.g. "buy BANK 10" -> ["buy", "bank", "10"]
        parts = command.lower().split()

        if len(parts) == 1 and parts[0] == "hold":
            action, ticker, quantity = "hold", "-", 0
            print("You sit on your hands.")
            break

        if len(parts) != 3 or parts[0] not in ["buy", "sell"]:
            print("Not a valid move. Try:  buy BANK 10")
            if AUTO_PLAY:
                action, ticker, quantity = "hold", "-", 0
                break
            continue

        action   = parts[0]
        ticker   = parts[1].upper()
        quantity = parts[2]

        if ticker not in assets:
            print(f"There is no {ticker} on this desk.")
            if AUTO_PLAY:
                action, ticker, quantity = "hold", "-", 0
                break
            continue

        # .isdigit() is a string method: True only if every character is a digit, so it
        # is safe to cast straight to int() afterwards (casting, Class 1)
        if not quantity.isdigit() or int(quantity) == 0:
            print("Quantity must be a whole number above zero.")
            if AUTO_PLAY:
                action, ticker, quantity = "hold", "-", 0
                break
            continue

        quantity = int(quantity)
        price    = assets[ticker]["price"]
        value    = round(quantity * price, 2)

        if action == "buy" and value > cash:
            print(f"That costs {value:,.2f} and you only have {cash:,.2f}.")
            if AUTO_PLAY:
                action, ticker, quantity = "hold", "-", 0
                break
            continue

        if action == "sell" and quantity > portfolio[ticker]:
            print(f"You only hold {portfolio[ticker]} {ticker}.")
            if AUTO_PLAY:
                action, ticker, quantity = "hold", "-", 0
                break
            continue

        if action == "buy":
            cash = round(cash - value, 2)
            portfolio[ticker] = portfolio[ticker] + quantity
            print(f"BOUGHT {quantity} {ticker} at {price:,.2f} = {value:,.2f}")
        else:
            cash = round(cash + value, 2)
            portfolio[ticker] = portfolio[ticker] - quantity
            print(f"SOLD   {quantity} {ticker} at {price:,.2f} = {value:,.2f}")

        trade_log.append({
            "month": month, "action": action, "ticker": ticker,
            "quantity": quantity, "price": price, "value": value,
        })
        break

    holdings_value = 0
    for ticker in portfolio:
        holdings_value = holdings_value + portfolio[ticker] * assets[ticker]["price"]

    benchmark = 0
    for ticker in benchmark_units:
        benchmark = benchmark + benchmark_units[ticker] * assets[ticker]["price"]

    history.append({
        "month": month,
        "cash": round(cash, 2),
        "holdings": round(holdings_value, 2),
        "net_worth": round(cash + holdings_value, 2),
        "benchmark": round(benchmark, 2),
    })

    month = month + 1

print("")
print("=" * 62)
print("TRADING CLOSED")
print("=" * 62)

final = history[-1]
print(f"start      {START_CASH:>12,.2f}")
print(f"you        {final['net_worth']:>12,.2f}")
print(f"benchmark  {final['benchmark']:>12,.2f}")
print(f"trades     {len(trade_log)}")

---
## 4. 🔍 The debrief — now with pandas

`trade_log` is a list of dictionaries — one dict per trade, all with the same keys. That
is *exactly* the shape pandas wants: hand it straight to `pd.DataFrame()` and every trade
becomes a row, every key becomes a column. This is the pivot Class 3 was building
towards — the loop above did the work, pandas now makes sense of it.

In [ ]:
# A list of dicts becomes a DataFrame in one line (Class 3)
trades = pd.DataFrame(trade_log)

print(trades.shape)     # (rows, columns)
trades.info()           # types, and how many trades were actually logged
trades.head()           # the first 5 trades

In [ ]:
# How much money moved through each action, and which ticker got the most attention
print(trades.groupby("action")["value"].sum())
print("")
print(trades["ticker"].value_counts())

# Read together: if "buy" outweighs "sell", you finished the year holding a bigger book
# than you started with. Whichever ticker tops value_counts() is the one you traded the
# most *often* - not necessarily the one that made you the most money (see below).

In [ ]:
# Work out the month-over-month change with a plain loop before handing it to pandas.
# enumerate() hands us the position and the month's dictionary together (Class 2).
changes = []
previous_net_worth = START_CASH
for index, month_data in enumerate(history):
    change = round(month_data["net_worth"] - previous_net_worth, 2)
    changes.append(change)
    previous_net_worth = month_data["net_worth"]

hist = pd.DataFrame(history)
hist["change"] = changes          # a calculated column, this time built by a loop
print(hist)

# Best and worst month - boolean filtering against .max() / .min(), Class 3 style,
# no need for anything fancier.
best_change  = hist["change"].max()
worst_change = hist["change"].min()
best_month   = hist[hist["change"] == best_change]
worst_month  = hist[hist["change"] == worst_change]

print("")
print("Best month:")
print(best_month)
print("")
print("Worst month:")
print(worst_month)

### 📊 One row per asset — a loop collected it, pandas summarised it

For each ticker we walk `trade_log` with a plain `for` loop, add up what was spent and
what was received, then work out the result including whatever is still held at today's
price. Once we have one dictionary per ticker, `pd.DataFrame(rows)` and `.sort_values()`
do the rest — a loop built the numbers, pandas presents them. That is the whole workflow
this course teaches, one more time.

In [ ]:
rows = []
for ticker in assets:
    bought = 0
    sold   = 0
    for trade in trade_log:
        if trade["ticker"] == ticker and trade["action"] == "buy":
            bought = bought + trade["value"]
        if trade["ticker"] == ticker and trade["action"] == "sell":
            sold = sold + trade["value"]
    still_held = portfolio[ticker] * assets[ticker]["price"]
    rows.append({
        "ticker": ticker, "spent": round(bought, 2), "received": round(sold, 2),
        "still_held": round(still_held, 2),
        "result": round(sold + still_held - bought, 2),
    })

result_table = pd.DataFrame(rows).sort_values("result", ascending=False)
print(result_table)

### 💾 Exporting — the Class 3 finishing move

Any DataFrame can be handed to `.to_csv()` and saved as a file someone else can open in
Excel — the same finishing move as the Class 3 monthly expenses report.

In [ ]:
result_table.to_csv("trading_results.csv", index=False)
print("Saved trading_results.csv")

---
### 🏁 The verdict

We already have everything we need in `final` and `START_CASH` — one `if` / `elif` /
`else` turns the numbers into a sentence, written so it reads correctly whichever way the
numbers land (nobody hardcoded who wins).

In [ ]:
final            = history[-1]
your_result      = round(final["net_worth"] - START_CASH, 2)
benchmark_result = round(final["benchmark"] - START_CASH, 2)

print(f"You:        {final['net_worth']:>12,.2f} EUR   (vs start: {your_result:>10,.2f})")
print(f"Benchmark:  {final['benchmark']:>12,.2f} EUR   (vs start: {benchmark_result:>10,.2f})")
print("")

if final["net_worth"] > final["benchmark"]:
    print("VERDICT: you beat the benchmark. Well played.")
elif final["net_worth"] == final["benchmark"]:
    print("VERDICT: dead heat with the benchmark.")
else:
    print("VERDICT: the benchmark beat you. Buying everything on day one and doing")
    print("         nothing would have paid more - a very real lesson about trying too hard.")

if final["net_worth"] > START_CASH:
    print("At least you finished above your starting cash.")
elif final["net_worth"] == START_CASH:
    print("You finished exactly where you started - a long way to go nowhere.")
else:
    print("You finished below your starting cash.")

> 🆕 **Level 2 territory: one matplotlib chart, just for fun**
>
> Plotting is not part of this course — Level 2 covers it properly. But a pandas
> DataFrame knows how to draw itself with `.plot()`, and it is one line, so here it is:
> your net worth against the benchmark, month by month.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt   # 🆕 not covered in this course - Level 2 territory

hist.plot(x="month", y=["net_worth", "benchmark"], marker="o", figsize=(9, 5),
          title="Your net worth vs the passive benchmark")
plt.ylabel("EUR")
plt.show()

---
## 5. 🗺️ Where you learned each piece

| Mechanic in the game | What it actually is | Where you learned it |
|---|---|---|
| Settings (`AUTO_PLAY`, `SEED`, `START_CASH`...) | Variables | Class 1 |
| `random.seed`, `random.choice`, `random.uniform` | A new library, used just like `pandas` | 🆕 today |
| `assets` | A dictionary of dictionaries, nested one level | Class 2 |
| `headlines` and tuple unpacking | A list of tuples | Class 2 |
| `portfolio`, `trade_log`, `history` | Dictionaries and lists | Class 2 |
| The price board, `{price:>9,.2f}` | f-strings with format specifiers | Class 1 |
| 🟢▲ / 🔴▼ up-or-down decision | `if` / `else` | Class 1 |
| The 12-month loop | `while` | Class 1 |
| Looping over assets / portfolio / trade_log | `for` | Class 1 and 2 |
| The move-validation loop | `while True` with `break` / `continue` | Class 1 |
| `.split()`, `.isdigit()` | String methods | Class 1 |
| Casting the quantity to `int()` | Casting | Class 1 |
| `ticker not in assets`, `parts[0] not in [...]` | Membership with `in` / `not in` | Class 1 |
| `enumerate(history)` | A built-in function | Class 2 |
| `pd.DataFrame(trade_log)`, `.head()`, `.info()`, `.shape` | Loading and inspecting a DataFrame | Class 3 |
| `.groupby("action")["value"].sum()` | Grouping and summarising | Class 3 |
| `.value_counts()` | Counting categories | Class 3 |
| The `change` column, boolean filtering vs `.max()` / `.min()` | Calculated columns and filtering | Class 3 |
| `.sort_values("result", ascending=False)` | Sorting a DataFrame | Class 3 |
| `.to_csv("trading_results.csv", index=False)` | Exporting | Class 3 |
| The final `if` / `elif` / `else` verdict | Decisions | Class 1 |
| The matplotlib chart | Plotting | 🆕 Level 2 |

Nothing above was invented for this demo. It is Classes 1 to 3, arranged into something
that runs.

---
## 6. 🚀 Make it yours

The whole point of a script instead of a black box: you can open it up and change it. Try
one of these, then re-run **from the settings cell down** (remember: `assets` gets
mutated by the loop, and `original_prices` only exists to undo that):

1. **Add a fifth asset** to the `assets` dictionary and give it a `swing`. Does it show up
   everywhere on its own, or do you need to touch `portfolio` too? (Hint: yes — `portfolio`
   is a separate dictionary.)
2. **Make the swings bigger or smaller** and see how much noisier — or calmer — the price
   board gets, and what that does to the final verdict.
3. **Add a transaction fee**: subtract a fixed EUR amount, or a percentage of `value`,
   every time a trade goes through.
4. **Write a smarter auto-pilot**: instead of a fixed `script` list, add an `if` inside
   the move-picking loop that reacts to the news (e.g. buy whichever ticker the headline
   just hit).
5. **Extend the run**: change `MONTHS` to 24 and see whether the benchmark's lead grows,
   shrinks, or flips.

Set `AUTO_PLAY = False` in the settings cell to play it yourself instead of watching the
script — Colab will give you an `input()` box under the game cell.